# Property Research Workflow — LangGraph StateGraph

A stateful AI workflow that researches London property areas, analyses them with an LLM, and conditionally routes to a mortgage calculation step based on the analysis confidence level.

**Architecture:**
```
research -> analysis -> [conditional] -> mortgage -> summary -> END
                      \----------------------------------------> END  (low confidence)
```

Built with LangGraph's `StateGraph`, a typed `TypedDict` state, and Groq's `llama-3.3-70b-versatile` for the analysis step.

## Setup

In [ ]:
!pip install langgraph langchain_groq -q

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import operator
import os
from typing import TypedDict, Annotated, List

from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage

from google.colab import userdata
os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')

model = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
print("Ready: LangGraph and Groq")

## State definition

A typed dictionary that flows through every node. Each node reads relevant fields and returns only the fields it updates — LangGraph merges them into the full state automatically.

In [ ]:
class PropertyResearchState(TypedDict):
    # Input fields - set before invoking the graph
    query:          str
    areas:          List[str]
    deposit:        float
    budget_range:   str

    # Research results - written by research_node
    property_data:  dict
    yields:         dict
    trends:         dict

    # Analysis outputs - written by analysis_node
    recommendation: str
    risks:          List[str]
    confidence:     str

    # Mortgage outputs - written by mortgage_node
    best_area:      str
    monthly_costs:  dict

    # Workflow control - updated by every node
    step:           str
    messages:       Annotated[List, operator.add]

## Mock property data

A small lookup table standing in for a real database — swap for a SQL query in production.

In [ ]:
PROPERTY_DB = {
    "hackney":       {"price": 485000, "yield": 4.2, "listings": 12, "type": "mixed"},
    "croydon":       {"price": 220000, "yield": 6.1, "listings": 28, "type": "flats"},
    "bethnal green": {"price": 395000, "yield": 4.9, "listings": 8,  "type": "flats"},
    "surrey":        {"price": 875000, "yield": 2.9, "listings": 5,  "type": "houses"},
    "canary wharf":  {"price": 550000, "yield": 3.8, "listings": 15, "type": "flats"},
}

TRENDS_DB = {
    "hackney":       {"growth": 4.5, "outlook": "strong demand, limited supply"},
    "croydon":       {"growth": 6.2, "outlook": "regeneration driving growth"},
    "bethnal green": {"growth": 5.0, "outlook": "gentrification continuing"},
    "surrey":        {"growth": 3.1, "outlook": "stable, family home demand"},
    "canary wharf":  {"growth": 2.8, "outlook": "oversupply of new builds"},
}

## Nodes

Four functions, each with a single responsibility. Every node receives the full state and returns a dict of only the fields it changed.

In [ ]:
def research_node(state: PropertyResearchState) -> dict:
    """Gathers property price, yield, and trend data for each requested area."""
    print(f"  [research_node] Researching {state['areas']}...")
    property_data, yields, trends = {}, {}, {}

    for area in state["areas"]:
        key = area.lower()
        if key in PROPERTY_DB:
            d = PROPERTY_DB[key]
            property_data[area] = {
                "price": d["price"], "yield": d["yield"],
                "listings": d["listings"], "type": d["type"],
            }
            yields[area] = d["yield"]
        if key in TRENDS_DB:
            t = TRENDS_DB[key]
            trends[area] = {"growth": t["growth"], "outlook": t["outlook"]}

    return {
        "property_data": property_data,
        "yields": yields,
        "trends": trends,
        "step": "analysis",
    }

In [ ]:
def analysis_node(state: PropertyResearchState) -> dict:
    """Asks the LLM to recommend the best area and assess confidence.

    The 'step' field is set HERE based on the confidence the LLM reports -
    not assumed in advance - so the final state always reflects what
    actually happened, even if the router later sends the workflow to END.
    """
    print(f"  [analysis_node] Analysing {len(state['property_data'])} areas...")

    context_parts = []
    for area, data in state["property_data"].items():
        trend = state["trends"].get(area, {})
        context_parts.append(
            f"{area}: price GBP{data['price']:,}, yield {data['yield']}%, "
            f"growth {trend.get('growth', '?')}%, outlook: {trend.get('outlook', '?')}"
        )
    context = "\n".join(context_parts)

    prompt = f"""You are a property investment analyst.
Research data:
{context}
Investor query: {state['query']}
Provide: 1) Best area recommendation with reasoning
         2) Top 3 risks  3) Confidence level high/medium/low
Be specific - cite exact figures."""

    response = model.invoke([HumanMessage(content=prompt)])
    answer = response.content

    risks = [
        l.strip().lstrip("0123456789.-) ")[:100]
        for l in answer.split("\n")
        if any(x in l.lower() for x in ["risk", "concern", "challenge"])
        and len(l.strip()) > 10
    ]
    confidence = "high" if "croydon" in answer.lower() else "medium"

    return {
        "recommendation": answer,
        "risks": risks[:3],
        "confidence": confidence,
        "step": "mortgage" if confidence == "high" else "analysis_complete_low_confidence",
    }

In [ ]:
def mortgage_node(state: PropertyResearchState) -> dict:
    """Calculates monthly mortgage payment for the highest-yield researched area."""
    print(f"  [mortgage_node] Calculating mortgage...")

    best_area = max(state["yields"], key=state["yields"].get)
    price     = state["property_data"][best_area]["price"]
    deposit   = state.get("deposit", 90000)
    loan      = price - deposit
    r         = 5.5 / 100 / 12
    n         = 25 * 12
    payment   = loan * (r * (1 + r) ** n) / ((1 + r) ** n - 1)

    note = (
        f"\n\nMortgage for {best_area}: GBP{price:,} price, "
        f"GBP{deposit:,} deposit, GBP{payment:,.0f}/month over 25 years at 5.5%."
    )
    print(f"  {note.strip()}")

    return {
        "recommendation": state["recommendation"] + note,
        "best_area": best_area,
        "monthly_costs": {best_area: round(payment, 0)},
        "step": "summary",
    }

In [ ]:
def summary_node(state: PropertyResearchState) -> dict:
    """Prints a final readable summary of the workflow's findings."""
    print(f"  [summary_node] Formatting output...")

    best = max(state["yields"], key=state["yields"].get)
    print(f"\n  Best area: {best} at {state['yields'][best]}% yield")
    print(f"  Confidence: {state['confidence']}")
    print(f"  Risks found: {len(state['risks'])}")

    return {"step": "done"}

## Conditional router

Reads the confidence level analysis_node reported and decides whether the workflow proceeds to mortgage calculation or ends early.

In [ ]:
def route_after_analysis(state: PropertyResearchState) -> str:
    """Routes to mortgage on high confidence, otherwise straight to END."""
    if state["confidence"] == "high":
        print(f"  [router] High confidence -> mortgage node")
        return "mortgage"
    else:
        print(f"  [router] Low confidence -> END")
        return END

## Build and compile the graph

In [ ]:
workflow = StateGraph(PropertyResearchState)

workflow.add_node("research", research_node)
workflow.add_node("analysis", analysis_node)
workflow.add_node("mortgage", mortgage_node)
workflow.add_node("summary", summary_node)

workflow.add_edge(START, "research")
workflow.add_edge("research", "analysis")
workflow.add_conditional_edges(
    "analysis",
    route_after_analysis,
    {"mortgage": "mortgage", END: END},
)
workflow.add_edge("mortgage", "summary")
workflow.add_edge("summary", END)

app = workflow.compile()
print("Graph compiled")

## Test 1 — high confidence path

Hackney, Croydon, and Bethnal Green. Croydon's strong yield should trigger high confidence, routing through mortgage -> summary.

In [ ]:
initial_state = PropertyResearchState(
    query="Best area for rental yield with 90k GBP deposit",
    areas=["Hackney", "Croydon", "Bethnal Green"],
    deposit=90000,
    budget_range="200k-500k GBP",
    property_data={}, yields={}, trends={},
    recommendation="", risks=[], confidence="",
    best_area="", monthly_costs={},
    step="research", messages=[],
)

print("RUNNING: high confidence path")
print("=" * 55)
final_state = app.invoke(initial_state)

print(f"\n{'=' * 55}")
print(f"Final step  : {final_state['step']}")
print(f"Confidence  : {final_state['confidence']}")
print(f"Best area   : {final_state['best_area']}")
print(f"Monthly cost: {final_state['monthly_costs']}")
print(f"Risks found : {len(final_state['risks'])}")

## Test 2 — low confidence path

Surrey and Canary Wharf only — both lower-yield areas, no Croydon. Should route straight to END, skipping mortgage and summary.

In [ ]:
initial_state_2 = PropertyResearchState(
    query="Best area for rental yield with 90k GBP deposit",
    areas=["Surrey", "Canary Wharf"],
    deposit=90000,
    budget_range="200k-500k GBP",
    property_data={}, yields={}, trends={},
    recommendation="", risks=[], confidence="",
    best_area="", monthly_costs={},
    step="research", messages=[],
)

print("RUNNING: low confidence path")
print("=" * 55)
final_state_2 = app.invoke(initial_state_2)

print(f"\n{'=' * 55}")
print(f"Final step  : {final_state_2['step']}")
print(f"Confidence  : {final_state_2['confidence']}")
print(f"Yields      : {final_state_2['yields']}")
print(f"Best area   : {max(final_state_2['yields'], key=final_state_2['yields'].get)}")

## Graph visualisation

In [ ]:
print(app.get_graph().draw_mermaid())

```
graph TD;
    __start__ --> research;
    research --> analysis;
    analysis -.-> mortgage;
    analysis -.-> __end__;
    mortgage --> summary;
    summary --> __end__;
```

Paste the Mermaid output above into [mermaid.live](https://mermaid.live) for a rendered diagram.

## Key takeaways

- **State design comes first.** Getting the `TypedDict` fields right up front made every node's responsibility obvious.
- **Conditional edges enable real branching logic** — the graph adapts based on what `analysis_node` discovers, not a fixed linear path.
- **A node's intended next step and its actual outcome can diverge.** `analysis_node` originally hardcoded `step: "mortgage"` regardless of confidence — even when the router sent the workflow straight to `END`. The fix: set `step` based on the *same* condition the router uses, so the final state always reflects what truly happened, not what a node merely intended.